In [ ]:
# ============================================================
# 01_final_data_preparation
# Thesis final pipeline
# ============================================================

import re
import sys
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)

PROJECT_CONSTANTS_DIR = Path(
    "/home/azureuser/cloudfiles/code/Users/m.bolognini/"
    "UseCase-Code/data-science/projects/a_MB_thesis_final/pipeline_thesis_final"
)

sys.path.append(str(PROJECT_CONSTANTS_DIR))

from project_constants import *

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_01:", OUTPUT_01)

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_str_upper(series):
    return series.astype("string").str.strip().str.upper()


def clean_str_lower(series):
    return series.astype("string").str.strip().str.lower()


def make_episode_key(pubid, episode_id):
    return (
        pubid.astype(str).str.strip().str.upper()
        + "__"
        + episode_id.astype(str).str.strip().str.upper()
    )


def standardize_episode_key(series):
    return (
        series.astype(str)
        .str.strip()
        .str.upper()
        .str.replace(
            r"^(P_[A-Z0-9]+)_([AB]\d+)$",
            r"\1__\2",
            regex=True
        )
    )


def parse_date_column(series):
    raw = (
        series.astype(str)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "NaT": np.nan,
            "None": np.nan,
            "NULL": np.nan,
            "<NA>": np.nan
        })
    )

    return pd.to_datetime(
        raw,
        errors="coerce",
        format="mixed",
        dayfirst=True
    )


def make_parquet_safe(df):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype("string")
    return df


def save_parquet(df, path):
    df_safe = make_parquet_safe(df)
    df_safe.to_parquet(path, index=False)
    print("Saved:", path)

In [ ]:
# ============================================================
# LOAD RAW DATASETS
# ============================================================

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# -----------------------------
# CGP raw files - 15 Aprile
# -----------------------------

main_raw_path = RAW_DIR / "15 Aprile" / "med_cli_main_15_04_2026.xlsx"
episodes_raw_path = RAW_DIR / "15 Aprile" / "med_cli_ac_episodes_15_04_2026.xlsx"
adm_reason_raw_path = RAW_DIR / "15 Aprile" / "med_cli_admissReason_15_04_2026.xlsx"

# -----------------------------
# Episode master source files - 21 Aprile
# -----------------------------

RAW_21_DIR = RAW_DIR / "21 Aprile"

episode_master_raw_files = {
    "adm_devices": RAW_21_DIR / "med_cli_acEpisodes_adm_devices_21_04_2026.xlsx",
    "admc_diagnostics": RAW_21_DIR / "med_cli_acEpisodes_admc_diagnostics_21_04_2026.xlsx",
    "nosoc_inf_etiology": RAW_21_DIR / "med_cli_acEpisodes_admc_nosocInfEtiol_21_04_2026.xlsx",
    "agent": RAW_21_DIR / "med_cli_acEpisodes_agent_21_04_2026.xlsx",
    "discharge_icd10": RAW_21_DIR / "med_cli_acEpisodes_disch_ICD10_21_04_2026.xlsx",
    "int_devices": RAW_21_DIR / "med_cli_acEpisodes_int_devices_21_04_2026.xlsx",
    "other_invasive_procedures": RAW_21_DIR / "med_cli_acEpisodes_int_othInvProcList_21_04_2026.xlsx",
    "adm_reason": RAW_21_DIR / "med_cli_admReason_21_04_2026.xlsx",
    "episodes": RAW_21_DIR / "piv_med_cli_ac_episodes_21_04_2026.xlsx",
    "hema": RAW_21_DIR / "piv_med_cli_hemaExams_21_04_2026.xlsx",
    "main": RAW_21_DIR / "piv_med_cli_main_21_04_2026.xlsx",
}

# -----------------------------
# Mapping
# -----------------------------

mapping_path = RAW_DIR / "Mapping_pubID_episodekey_CPI_endDate.xlsx"

# -----------------------------
# Raw labs
# -----------------------------

labs_raw_path = RAW_DIR / "labs_clean_source_copy.parquet"

# -----------------------------
# Death dataset
# -----------------------------

death_path = PROCESSED_DIR / "new_df_episodekey_deathdate.csv"

# -----------------------------
# Solid Tumor
# -----------------------------

solid_tumor_path = PROCESSED_DIR / "df_solidTumor.csv"

df_main_raw = pd.read_excel(main_raw_path)
df_episodes_raw = pd.read_excel(episodes_raw_path)
df_adm_reason_raw = pd.read_excel(adm_reason_raw_path)
df_mapping_raw = pd.read_excel(mapping_path)
df_labs_raw = pd.read_parquet(labs_raw_path)
df_death_raw = pd.read_csv(death_path)
df_solid_raw = pd.read_csv(solid_tumor_path)

for d in [
    df_main_raw,
    df_episodes_raw,
    df_adm_reason_raw,
    df_mapping_raw,
    df_death_raw,
    df_solid_raw
]:
    d.columns = d.columns.str.strip()

df_labs_raw.columns = df_labs_raw.columns.str.strip()

print("df_main_raw:", df_main_raw.shape)
print("df_episodes_raw:", df_episodes_raw.shape)
print("df_adm_reason_raw:", df_adm_reason_raw.shape)
print("df_mapping_raw:", df_mapping_raw.shape)
print("df_labs_raw:", df_labs_raw.shape)
print("df_death_raw:", df_death_raw.shape)
print("df_solid_raw:", df_solid_raw.shape)


In [ ]:
# ============================================================
# BUILD 01_episode_master FROM CLEAN PIVOT FILES
# ============================================================

DF_CLEAN_PIV_DIR = PROCESSED_DIR / "df_clean_piv"

df_episode_master_base = pd.read_excel(DF_CLEAN_PIV_DIR / "df_episodes_long.xlsx")
df_main_master = pd.read_excel(DF_CLEAN_PIV_DIR / "df_main_long.xlsx")

for d in [df_episode_master_base, df_main_master]:
    d.columns = d.columns.str.strip()

df_episode_master = df_episode_master_base.copy()
df_main_master = df_main_master.copy()

df_episode_master = df_episode_master.rename(columns={"episodeId": "episodeID"})
df_main_master = df_main_master.rename(columns={"episodeId": "episodeID"})

df_episode_master["pubID"] = clean_str_upper(df_episode_master["pubID"])
df_episode_master["episodeID"] = clean_str_upper(df_episode_master["episodeID"])
df_episode_master["hospitalWard"] = clean_str_upper(df_episode_master["hospitalWard"])

df_main_master["pubID"] = clean_str_upper(df_main_master["pubID"])
df_main_master["episodeID"] = clean_str_upper(df_main_master["episodeID"])
df_main_master["hospitalWard"] = clean_str_upper(df_main_master["hospitalWard"])

df_episode_master["startDate"] = parse_date_column(df_episode_master["startDate"])
df_episode_master["endDate"] = parse_date_column(df_episode_master["endDate"])

# Merge patient-level/main variables
df_episode_master = df_episode_master.merge(
    df_main_master,
    on=["pubID", "episodeID", "hospitalWard"],
    how="left",
    suffixes=("", "_main")
)

df_episode_master["episode_key"] = make_episode_key(
    df_episode_master["pubID"],
    df_episode_master["episodeID"]
)

print("Episode master after main merge:", df_episode_master.shape)
print("Unique episodes:", df_episode_master["episode_key"].nunique())
print("Duplicated episode_key:", df_episode_master["episode_key"].duplicated().sum())

assert df_episode_master["episode_key"].duplicated().sum() == 0

In [ ]:
# ============================================================
# LEAKAGE NOTE
# ============================================================

# Some variables included in 01_episode_master, such as discharge diagnoses,
# nosocomial infection etiologies, diagnostic summaries, and aetiological agents,
# are post-admission or post-discharge descriptors.
#
# These variables are retained for descriptive characterization and quality control
# only. They are intentionally excluded from all supervised prediction and survival
# modelling feature sets to avoid temporal leakage.

In [ ]:
# ============================================================
# ADD CLOUDPATIENTID FROM MAPPING
# ============================================================

df_mapping = df_mapping_raw.copy()
df_mapping.columns = df_mapping.columns.str.strip()

df_mapping["CloudPatientId"] = clean_str_lower(df_mapping["CloudPatientId"])
df_mapping["episode_key"] = standardize_episode_key(df_mapping["episode_key"])

df_mapping_episode = (
    df_mapping[["episode_key", "CloudPatientId"]]
    .drop_duplicates()
    .copy()
)

print("Mapping rows:", df_mapping_episode.shape[0])
print("Duplicated episode_key in mapping:", df_mapping_episode["episode_key"].duplicated().sum())

assert df_mapping_episode["episode_key"].duplicated().sum() == 0

df_episode_master = df_episode_master.merge(
    df_mapping_episode,
    on="episode_key",
    how="left"
)

print("Missing CloudPatientId in episode master:",
      df_episode_master["CloudPatientId"].isna().sum())

In [ ]:
# ============================================================
# ADD AGGREGATED EPISODE-LEVEL VARIABLES
# ============================================================

def clean_episode_keys_from_raw(df, pub_col="pubID", ep_col="_acEpisodes_key"):
    df = df.copy()
    df[pub_col] = clean_str_upper(df[pub_col])
    df[ep_col] = clean_str_upper(df[ep_col])
    df = df.rename(columns={ep_col: "episodeID"})
    return df


def build_episode_concat_value(df, feature_name):
    df = clean_episode_keys_from_raw(df)
    df["value"] = df["value"].astype(str).str.strip()
    df = df.loc[df["value"].notna() & (df["value"] != "") & (df["value"] != "nan")].copy()
    df = df.drop_duplicates(["pubID", "episodeID", "value"])

    out = (
        df.groupby(["pubID", "episodeID"])["value"]
        .apply(lambda x: " | ".join(sorted(set(x))))
        .reset_index()
        .rename(columns={"value": feature_name})
    )
    return out


def build_episode_concat_icd(df, feature_name="discharge_icd10"):
    df = clean_episode_keys_from_raw(df)
    df["diseaseClass"] = df["diseaseClass"].astype(str).str.strip()
    df["disease"] = df["disease"].astype(str).str.strip()
    df["disease_full"] = df["diseaseClass"] + " - " + df["disease"]
    df = df.drop_duplicates(["pubID", "episodeID", "disease_full"])

    out = (
        df.groupby(["pubID", "episodeID"])["disease_full"]
        .apply(lambda x: " | ".join(sorted(set(x))))
        .reset_index()
        .rename(columns={"disease_full": feature_name})
    )
    return out


def safe_merge_episode(base_df, new_df, name):
    dup = new_df.duplicated(["pubID", "episodeID"]).sum()
    print(f"{name} duplicates:", dup)
    assert dup == 0, f"{name} has duplicated pubID + episodeID"

    return base_df.merge(
        new_df,
        on=["pubID", "episodeID"],
        how="left"
    )


df_adm_reason_21 = pd.read_excel(episode_master_raw_files["adm_reason"])
df_adm_devices = pd.read_excel(episode_master_raw_files["adm_devices"])
df_admc_diagnostics = pd.read_excel(episode_master_raw_files["admc_diagnostics"])
df_nosoc_inf_etio = pd.read_excel(episode_master_raw_files["nosoc_inf_etiology"])
df_agent = pd.read_excel(episode_master_raw_files["agent"])
df_int_devices = pd.read_excel(episode_master_raw_files["int_devices"])
df_int_oth_inv = pd.read_excel(episode_master_raw_files["other_invasive_procedures"])
df_disch_icd10 = pd.read_excel(episode_master_raw_files["discharge_icd10"])

for d in [
    df_adm_reason_21,
    df_adm_devices,
    df_admc_diagnostics,
    df_nosoc_inf_etio,
    df_agent,
    df_int_devices,
    df_int_oth_inv,
    df_disch_icd10,
]:
    d.columns = d.columns.str.strip()

aggregates = {
    "adm_reason": build_episode_concat_value(df_adm_reason_21, "adm_reason"),
    "adm_devices": build_episode_concat_value(df_adm_devices, "adm_devices"),
    "admc_diagnostics": build_episode_concat_value(df_admc_diagnostics, "admc_diagnostics"),
    "nosoc_inf_etiology": build_episode_concat_value(df_nosoc_inf_etio, "nosoc_inf_etiology"),
    "aetiological_agents": build_episode_concat_value(df_agent, "aetiological_agents"),
    "int_devices": build_episode_concat_value(df_int_devices, "int_devices"),
    "other_invasive_procedures": build_episode_concat_value(df_int_oth_inv, "other_invasive_procedures"),
    "discharge_icd10": build_episode_concat_icd(df_disch_icd10, "discharge_icd10"),
}

for name, agg in aggregates.items():
    df_episode_master = safe_merge_episode(df_episode_master, agg, name)

print("Episode master after aggregated variables:", df_episode_master.shape)

In [ ]:
# ============================================================
# CREATE 01_death AND MERGE INTO EPISODE MASTER
# ============================================================

df_death = df_death_raw.copy()
df_death.columns = df_death.columns.str.strip()

df_death["episode_key"] = standardize_episode_key(df_death["episode_key"])
df_death["dateOfDeath"] = parse_date_column(df_death["dateOfDeath"])

df_death = (
    df_death[["episode_key", "dateOfDeath"]]
    .dropna(subset=["dateOfDeath"])
    .sort_values(["episode_key", "dateOfDeath"])
    .drop_duplicates("episode_key", keep="last")
    .copy()
)

print("01_death rows:", df_death.shape[0])
print("Deaths with date:", df_death["dateOfDeath"].notna().sum())
print("Duplicated episode_key:", df_death["episode_key"].duplicated().sum())

assert df_death["episode_key"].duplicated().sum() == 0

df_episode_master = df_episode_master.drop(
    columns=[c for c in df_episode_master.columns if c == "dateOfDeath"],
    errors="ignore"
)

df_episode_master = df_episode_master.merge(
    df_death,
    on="episode_key",
    how="left"
)

print("Deaths in episode master:", df_episode_master["dateOfDeath"].notna().sum())

In [ ]:
# ============================================================
# FINALIZE 01_episode_master
# ============================================================

df_episode_master["daysInER"] = pd.to_numeric(
    df_episode_master["daysInER"],
    errors="coerce"
)

df_episode_master["LOS_days"] = (
    df_episode_master["endDate"] - df_episode_master["startDate"]
).dt.days + 1

# daysInER cannot logically exceed total LOS
df_episode_master["daysInER_capped"] = df_episode_master["daysInER"].clip(
    lower=0,
    upper=df_episode_master["LOS_days"]
)

df_episode_master["ER_last_day"] = (
    df_episode_master["startDate"]
    + pd.to_timedelta(df_episode_master["daysInER_capped"] - 1, unit="D")
)

df_episode_master["ward_startDate"] = (
    df_episode_master["startDate"]
    + pd.to_timedelta(df_episode_master["daysInER_capped"], unit="D")
)

# If the patient never reaches a ward day within the episode window,
# ward_startDate is set to NaT.
df_episode_master.loc[
    df_episode_master["ward_startDate"] > df_episode_master["endDate"],
    "ward_startDate"
] = pd.NaT

front_cols = [
    "pubID",
    "episodeID",
    "episode_key",
    "CloudPatientId",
    "hospitalWard",
    "epType",
    "origin",
    "startDate",
    "endDate",
    "LOS_days",
    "daysInER",
    "daysInER_capped",
    "ER_last_day",
    "ward_startDate",
    "dateOfDeath",
]

front_cols = [c for c in front_cols if c in df_episode_master.columns]
other_cols = [c for c in df_episode_master.columns if c not in front_cols]

df_episode_master = df_episode_master[front_cols + other_cols]

print("Final episode master:", df_episode_master.shape)
print("Unique episodes:", df_episode_master["episode_key"].nunique())
print("Duplicated episode_key:", df_episode_master["episode_key"].duplicated().sum())
print("Deaths with date:", df_episode_master["dateOfDeath"].notna().sum())
print("Negative LOS:", (df_episode_master["endDate"] < df_episode_master["startDate"]).sum())

assert df_episode_master["episode_key"].duplicated().sum() == 0

In [ ]:
# ============================================================
# QC: EPISODE DURATION AND ER/WARD TIMING
# ============================================================

print("LOS_days summary:")
display(df_episode_master["LOS_days"].describe())

print("\ndaysInER summary:")
display(df_episode_master["daysInER"].describe())

print("\nMissing timing variables:")
display(
    df_episode_master[
        ["startDate", "endDate", "LOS_days", "daysInER", "ER_last_day", "ward_startDate"]
    ].isna().sum()
)

print("\nTiming inconsistencies:")
print("LOS_days <= 0:", (df_episode_master["LOS_days"] <= 0).sum())
print("daysInER <= 0:", (df_episode_master["daysInER"] <= 0).sum())
print("ER_last_day < startDate:", (df_episode_master["ER_last_day"] < df_episode_master["startDate"]).sum())
print("ward_startDate < startDate:", (df_episode_master["ward_startDate"] < df_episode_master["startDate"]).sum())
print("ward_startDate > endDate:", (df_episode_master["ward_startDate"] > df_episode_master["endDate"]).sum())

display(
    df_episode_master.loc[
        (df_episode_master["LOS_days"] <= 0) |
        (df_episode_master["daysInER"] <= 0) |
        (df_episode_master["ward_startDate"] > df_episode_master["endDate"]),
        [
            "pubID",
            "episode_key",
            "startDate",
            "endDate",
            "LOS_days",
            "daysInER",
            "ER_last_day",
            "ward_startDate",
        ]
    ]
)

In [ ]:
# ============================================================
# CREATE 01_solidTumor
# ============================================================

# Prefer solidTumor from episode_master if present.
# Otherwise use df_solid_raw.
if "solidTumor" in df_episode_master.columns:
    df_solidTumor = df_episode_master[
        ["pubID", "episodeID", "episode_key", "solidTumor"]
    ].copy()
else:
    df_solid_raw["episode_key"] = standardize_episode_key(df_solid_raw["episode_key"])
    df_solidTumor = df_solid_raw.copy()

df_solidTumor["solidTumor"] = pd.to_numeric(
    df_solidTumor["solidTumor"],
    errors="coerce"
)

solid_tumor_map = {
    0: "No cancer",
    2: "Localized cancer",
    6: "Metastatic cancer",
}

df_solidTumor["solidTumor_cat"] = (
    df_solidTumor["solidTumor"]
    .map(solid_tumor_map)
    .fillna("Unknown")
)

df_solidTumor = df_solidTumor.drop_duplicates("episode_key").copy()

print("01_solidTumor:", df_solidTumor.shape)
print("Duplicated episode_key:", df_solidTumor["episode_key"].duplicated().sum())
print(df_solidTumor["solidTumor"].value_counts(dropna=False).sort_index())
print(df_solidTumor["solidTumor_cat"].value_counts(dropna=False))

assert df_solidTumor["episode_key"].duplicated().sum() == 0

In [ ]:
# ============================================================
# ADD SOLID TUMOR CATEGORY BACK TO EPISODE MASTER
# ============================================================

df_episode_master = df_episode_master.drop(
    columns=["solidTumor_cat"],
    errors="ignore"
)

df_episode_master = df_episode_master.merge(
    df_solidTumor[["episode_key", "solidTumor_cat"]],
    on="episode_key",
    how="left"
)

print("Missing solidTumor_cat in episode master:",
      df_episode_master["solidTumor_cat"].isna().sum())

print(df_episode_master["solidTumor_cat"].value_counts(dropna=False))

In [ ]:
# ============================================================
# CREATE 01_clinical
# ============================================================

clinical_rename = {
    "adm_ecog": "admECOG",
    "adm_barthelScore": "admBarthelScore",
    "adm_bradenScore": "admBradenScore",
    "cirs_totalScore": "cirsTotalScore",
    "admissNumb12m": "admLast12m",
    "admissNumb6m": "admLast6m",
    "dischargeMod": "dischargeMode",
    "adm_treatTarget": "treatTarget",
    "adm_reason": "admReason",
}

df_clinical = df_episode_master.rename(columns=clinical_rename).copy()

clinical_cols = [
    "episode_key",
    "CloudPatientId",
    "pubID",
    "episodeID",
    "sex",
    "hospitalWard",
    "age",
    "pksYear",
    "alcYears",
    "startDate",
    "endDate",
    "LOS_days",
    "daysInER",
    "daysInER_capped",
    "ER_last_day",
    "ward_startDate",
    "admLast12m",
    "admLast6m",
    "admECOG",
    "admBarthelScore",
    "admBradenScore",
    "cirsTotalScore",
    "admReason",
    "dischargedAlive",
    "dischargeMode",
    "treatTarget",
    "dateOfDeath",
    "solidTumor",
    "solidTumor_cat",
]

clinical_cols = [c for c in clinical_cols if c in df_clinical.columns]

df_clinical = df_clinical[clinical_cols].copy()

print("01_clinical:", df_clinical.shape)
print("Unique episodes:", df_clinical["episode_key"].nunique())

In [ ]:
# ============================================================
# CREATE 01_labs_selected
# ============================================================

df_labs = df_labs_raw.copy()
df_labs.columns = df_labs.columns.str.strip()

df_labs["Subject_identifier"] = clean_str_lower(df_labs["Subject_identifier"])
df_labs["Method_text"] = (
    df_labs["Method_text"]
    .astype(str)
    .str.strip()
    .str.replace("–", "-", regex=False)
    .str.replace(r"\s+", " ", regex=True)
)

df_labs["Date"] = pd.to_datetime(
    df_labs["Date"],
    errors="coerce",
    utc=True
).dt.tz_localize(None)

df_labs["Numeric_value"] = pd.to_numeric(
    df_labs["Numeric_value"],
    errors="coerce"
).astype("float64")

df_labs = df_labs.dropna(subset=["Date", "Numeric_value", "Method_text"]).copy()

selected_labs_map = {
    "S-PROTEINA C REATTIVA": "CRP",
    "EMOCROMO - Globuli bianchi": "WBC",
    "FORMULA - Neutrofili": "Neutrophils",
    "FORMULA - Linfociti": "Lymphocytes",
    "EMOCROMO - Emoglobina": "Hemoglobin",
    "PIASTRINE - Piastrine": "Platelets",
    "S-CREATININA": "Creatinine",
    "S-UREA": "Urea",
    "S-SODIO": "Sodium",
    "S-POTASSIO": "Potassium",
}

UNIT_RULES = {
    "FORMULA - Linfociti": {"keep_unit": "10^9/L"},
    "FORMULA - Neutrofili": {"keep_unit": "10^9/L"},
    "EMOCROMO - Emoglobina": {
        "keep_unit": "g/dL",
        "convert": [{"from_unit": "g/L", "to_unit": "g/dL", "func": lambda x: x / 10}]
    },
    "S-CREATININA": {
        "keep_unit": "mg/dL",
        "convert": [{"from_unit": "µmol/L", "to_unit": "mg/dL", "func": lambda x: x * 0.0113}]
    }
}

for test_name, rules in UNIT_RULES.items():
    if "convert" in rules:
        for conv in rules["convert"]:
            mask = (df_labs["Method_text"] == test_name) & (df_labs["Measure"] == conv["from_unit"])
            df_labs.loc[mask, "Numeric_value"] = df_labs.loc[mask, "Numeric_value"].apply(conv["func"])
            df_labs.loc[mask, "Measure"] = conv["to_unit"]

    keep_unit = rules["keep_unit"]
    wrong_unit = (df_labs["Method_text"] == test_name) & (df_labs["Measure"] != keep_unit)
    df_labs = df_labs.loc[~wrong_unit].copy()

df_labs_selected = df_labs[
    df_labs["Method_text"].isin(selected_labs_map.keys())
].copy()

df_labs_selected["lab_name"] = df_labs_selected["Method_text"].map(selected_labs_map)

# Use mapping raw for CloudPatientId + episode_key/endDate
episode_windows = df_episode_master[
    ["episode_key", "pubID", "episodeID", "CloudPatientId", "startDate", "endDate"]
].copy()

df_labs_selected = df_labs_selected.merge(
    episode_windows,
    left_on="Subject_identifier",
    right_on="CloudPatientId",
    how="inner"
)

df_labs_selected = df_labs_selected[
    (df_labs_selected["Date"].dt.normalize() >= df_labs_selected["startDate"].dt.normalize()) &
    (df_labs_selected["Date"].dt.normalize() <= df_labs_selected["endDate"].dt.normalize())
].copy()

# QC: check whether the same lab observation is assigned to multiple episodes
lab_overlap_qc = (
    df_labs_selected
    .assign(Date_day=df_labs_selected["Date"].dt.normalize())
    .groupby(
        ["Subject_identifier", "Date_day", "Method_text", "Numeric_value"],
        dropna=False
    )["episode_key"]
    .nunique()
    .reset_index(name="n_episode_keys")
)

potential_multi_episode_labs = lab_overlap_qc[
    lab_overlap_qc["n_episode_keys"] > 1
].copy()

print("Potential lab observations assigned to multiple episodes:")
print(potential_multi_episode_labs.shape[0])

display(potential_multi_episode_labs.head(20))

df_labs_selected["lab_day"] = (
    df_labs_selected["Date"].dt.normalize()
    - df_labs_selected["startDate"].dt.normalize()
).dt.days

print("01_labs_selected:", df_labs_selected.shape)
print("Episodes with labs:", df_labs_selected["episode_key"].nunique())
print("Patients with labs:", df_labs_selected["pubID"].nunique())
print(df_labs_selected["lab_name"].value_counts())

# ============================================================
# QC: SELECTED BIOMARKERS AND MEASUREMENT UNITS
# ============================================================

lab_unit_qc = (
    df_labs_selected
    .groupby(["lab_name", "Method_text", "Measure"], dropna=False)
    .size()
    .reset_index(name="n_observations")
    .sort_values(["lab_name", "n_observations"], ascending=[True, False])
)

display(lab_unit_qc)

In [ ]:
# ============================================================
# QC: EPISODES/PATIENTS WITHOUT SELECTED LABS
# ============================================================

episodes_with_selected_labs = set(df_labs_selected["episode_key"].unique())

no_selected_labs = df_episode_master[
    ~df_episode_master["episode_key"].isin(episodes_with_selected_labs)
].copy()

print("Episodes without selected labs:", no_selected_labs.shape[0])
print("Patients without selected labs:", no_selected_labs["pubID"].nunique())

display(
    no_selected_labs[
        [
            "pubID",
            "episode_key",
            "CloudPatientId",
            "hospitalWard",
            "startDate",
            "endDate",
            "LOS_days",
            "daysInER",
            "daysInER_capped",
        ]
    ]
)

In [ ]:
# ============================================================
# QC: WHY ARE SELECTED LABS MISSING?
# ============================================================

raw_lab_subjects = set(
    df_labs_raw["Subject_identifier"]
    .astype("string")
    .str.strip()
    .str.lower()
)

selected_lab_subjects_anytime = set(
    df_labs[
        df_labs["Method_text"].isin(selected_labs_map.keys())
    ]["Subject_identifier"]
    .astype("string")
    .str.strip()
    .str.lower()
)

no_selected_labs["has_any_raw_lab"] = (
    no_selected_labs["CloudPatientId"].isin(raw_lab_subjects)
)

no_selected_labs["has_selected_lab_anytime"] = (
    no_selected_labs["CloudPatientId"].isin(selected_lab_subjects_anytime)
)

display(
    no_selected_labs[
        [
            "pubID",
            "episode_key",
            "CloudPatientId",
            "has_any_raw_lab",
            "has_selected_lab_anytime",
            "startDate",
            "endDate",
        ]
    ]
)

print("\nSummary:")
display(
    no_selected_labs[
        ["has_any_raw_lab", "has_selected_lab_anytime"]
    ]
    .value_counts(dropna=False)
    .reset_index(name="n")
)

In [ ]:
# ============================================================
# QC: NEAREST SELECTED LABS FOR EPISODES WITHOUT IN-WINDOW LABS
# ============================================================

selected_labs_anytime = df_labs[
    df_labs["Method_text"].isin(selected_labs_map.keys())
].copy()

selected_labs_anytime["lab_name"] = selected_labs_anytime["Method_text"].map(selected_labs_map)

nearest_lab_check = selected_labs_anytime.merge(
    no_selected_labs[
        ["episode_key", "CloudPatientId", "startDate", "endDate"]
    ],
    left_on="Subject_identifier",
    right_on="CloudPatientId",
    how="inner"
)

nearest_lab_check["days_from_start"] = (
    nearest_lab_check["Date"].dt.normalize()
    - nearest_lab_check["startDate"].dt.normalize()
).dt.days

nearest_lab_check["days_from_end"] = (
    nearest_lab_check["Date"].dt.normalize()
    - nearest_lab_check["endDate"].dt.normalize()
).dt.days

nearest_lab_summary = (
    nearest_lab_check
    .sort_values(["episode_key", "days_from_start"])
    .groupby("episode_key")
    .agg(
        n_selected_labs_anytime=("lab_name", "size"),
        nearest_before_start_days=("days_from_start", lambda x: x[x < 0].max() if (x < 0).any() else np.nan),
        nearest_after_end_days=("days_from_end", lambda x: x[x > 0].min() if (x > 0).any() else np.nan),
        first_selected_lab_date=("Date", "min"),
        last_selected_lab_date=("Date", "max"),
    )
    .reset_index()
)

display(nearest_lab_summary)

In [ ]:
# ============================================================
# CREATE 01_LAB_MEASUREMENT_INTENSITY
# ============================================================

lab_intensity = (
    df_labs_selected
    .groupby("episode_key")
    .agg(
        total_lab_tests=("lab_name", "size"),
        n_distinct_biomarkers=("lab_name", "nunique"),
        first_lab_day=("lab_day", "min"),
        last_lab_day=("lab_day", "max"),
        n_lab_days=("lab_day", "nunique"),
    )
    .reset_index()
)

lab_days = (
    df_labs_selected[["episode_key", "lab_day"]]
    .drop_duplicates()
    .sort_values(["episode_key", "lab_day"])
    .copy()
)

lab_days["gap_from_previous_lab_day"] = (
    lab_days
    .groupby("episode_key")["lab_day"]
    .diff()
)

max_gap = (
    lab_days
    .groupby("episode_key")["gap_from_previous_lab_day"]
    .max()
    .reset_index(name="max_gap_between_lab_days")
)

lab_intensity = lab_intensity.merge(
    max_gap,
    on="episode_key",
    how="left"
)

lab_intensity["max_gap_between_lab_days"] = lab_intensity["max_gap_between_lab_days"].fillna(0)

print("01_lab_measurement_intensity:", lab_intensity.shape)
display(lab_intensity.describe())

In [ ]:
# ============================================================
# LABORATORY UNIT HARMONIZATION NOTE
# ============================================================

print("""
Laboratory unit harmonization:
- Hemoglobin values were harmonized to g/dL.
- Creatinine values were harmonized to mg/dL.
- Measurements with incompatible or non-interpretable units were excluded.
- Unit harmonization was performed before downstream feature engineering.
""")

In [ ]:
# ============================================================
# SAVE ALL OUTPUTS
# ============================================================

save_parquet(df_episode_master, OUTPUT_01 / "01_episode_master.parquet")
save_parquet(df_clinical, OUTPUT_01 / "01_clinical.parquet")
save_parquet(df_labs_selected, OUTPUT_01 / "01_labs_selected.parquet")
save_parquet(df_solidTumor, OUTPUT_01 / "01_solidTumor.parquet")
save_parquet(df_death, OUTPUT_01 / "01_death.parquet")
save_parquet(lab_intensity, OUTPUT_01 / "01_lab_measurement_intensity.parquet")

lab_unit_qc.to_excel(
    OUTPUT_01 / "01_lab_unit_qc.xlsx",
    index=False
)

print("All outputs saved in:", OUTPUT_01)

In [ ]:
# ============================================================
# FINAL QC
# ============================================================

print("=" * 80)
print("01 FINAL DATA PREPARATION SUMMARY")
print("=" * 80)

print("\n01_episode_master")
print(df_episode_master.shape)
print("Patients:", df_episode_master["pubID"].nunique())
print("Episodes:", df_episode_master["episode_key"].nunique())
print("Duplicated episode_key:", df_episode_master["episode_key"].duplicated().sum())
print("Deaths with date:", df_episode_master["dateOfDeath"].notna().sum())
print("Negative LOS:", (df_episode_master["endDate"] < df_episode_master["startDate"]).sum())
print("Missing CloudPatientId:", df_episode_master["CloudPatientId"].isna().sum())
print("Missing solidTumor_cat:", df_episode_master["solidTumor_cat"].isna().sum())

print("\n01_clinical")
print(df_clinical.shape)
print("Episodes:", df_clinical["episode_key"].nunique())
print("Missing CloudPatientId:", df_clinical["CloudPatientId"].isna().sum())

print("\n01_death")
print(df_death.shape)
print("Deaths with date:", df_death["dateOfDeath"].notna().sum())

print("\n01_solidTumor")
print(df_solidTumor.shape)
print(df_solidTumor["solidTumor"].value_counts(dropna=False).sort_index())

print("\n01_labs_selected")
print(df_labs_selected.shape)
print("Patients with labs:", df_labs_selected["pubID"].nunique())
print("Episodes with labs:", df_labs_selected["episode_key"].nunique())

print("\nTiming variables")
print("LOS_days <= 0:", (df_episode_master["LOS_days"] <= 0).sum())
print("ward_startDate > endDate:", (df_episode_master["ward_startDate"] > df_episode_master["endDate"]).sum())

print("\n01_lab_measurement_intensity")
print(lab_intensity.shape)
print("Episodes with lab intensity:", lab_intensity["episode_key"].nunique())

assert df_episode_master["episode_key"].duplicated().sum() == 0
assert df_death["episode_key"].duplicated().sum() == 0
assert df_solidTumor["episode_key"].duplicated().sum() == 0
assert (df_episode_master["endDate"] < df_episode_master["startDate"]).sum() == 0

print("\nAll checks passed.")